# Project 3: **Ask‑the‑Web Agent**

## Learning Objectives  
* Understand why tool calling is useful and how LLMs can invoke external tools.
* Implement a minimal loop that parses the LLM's output and executes a Python function.
* See how *function schemas* (docstrings and type hints) let us scale to many tools.
* Use **LangChain** to get function‑calling capability for free (ReAct reasoning, memory, multi‑step planning).
* Combine LLM with a web‑search tool to build a simple ask‑the‑web agent.

## Roadmap
1. Environment setup
2. Write simple tools and connect them to an LLM
3. Standardize tool calling by writing `to_schema`
4. Use LangChain to augment an LLM with your tools
5. Build a Perplexity‑style web‑search agent
6. (Optional) A minimal backend and frontend UI

In [1]:
# ---------------------------------------------------------
# Step 1: Implement the tool
# ---------------------------------------------------------

# def get_current_weather(city, unit = "celsius"):
#     return f"It is 23 {unit[0].upper()} and sunny in {city}"

# get_current_weather("San Diego")

def get_current_weather(city: str, unit: str = "celsius"):
    "Getting the current weather for a given city." 
    return f"It is 23 {unit[0].upper()} and sunny in {city}"

get_current_weather("San Diego")

'It is 23 C and sunny in San Diego'

In [2]:
# ---------------------------------------------------------
# Step 2: Create the prompt for the LLM to call tools
# ---------------------------------------------------------
SYSTEM_PROMPT=(
    "You are an assistant that can call tools. "
    "When the user asks something requiring fesh data, respond **only** with a JSON like: "
    'TOOL_CALL:{"name": <tool_name>, "args": { ... }}.'
)

TOOLS_SPEC = """
You can call exactly one tool:
- name: get_current_weather
  description: Return the current weather for a city.
  arguments:
    city: string
    unit: "celsius" | "fahrenheit" (optional, default "celsius")
"""
USER_QUESTION = "What is the weather in San Diego today?"

In [3]:
# ---------------------------------------------------------
# Step 3: Call the LLM with our prompt
# ---------------------------------------------------------
from openai import OpenAI

client = OpenAI(api_key = "ollama", base_url="http://localhost:11434/v1")

MODEL = "gemma3:1b"

response = client.chat.completions.create(
    model = MODEL,
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT + "\n\n" + TOOLS_SPEC},
        {"role": "user", "content": USER_QUESTION}
    ],
    temperature=0.1
)
raw = response.choices[0].message.content
print("\n Model output:\n", raw)


 Model output:
 TOOL_CALL:{"name": "get_current_weather", "args": { "city": "San Diego", "unit": "celsius" }}



In [4]:
# response

In [5]:
# ---------------------------------------------------------
# Step 4: Parse the LLM output and call the tool
# ---------------------------------------------------------
import re, json
pattern = r'TOOL_CALL:\s*({.*})'
match = re.search(pattern, raw, flags=re.DOTALL)

if match:
    call_json = json.loads(match.group(1))
    fn_name = call_json["name"]
    args = call_json.get("args", {})
    result = globals()[fn_name](**args)
    print("Result: ", result)
else:
    print("No tool call detected")

Result:  It is 23 C and sunny in San Diego


# Standadize tool calling

In [6]:
# ---------------------------------------------------------
# Generate a JSON schema for a tool automatically
# ---------------------------------------------------------
from pprint import pprint
import inspect

def to_schema(fn):
    sig = inspect.signature(fn)
    props = {
        name: {
            'type': 'string' if param.annotation is str else 'number',
            'description': f"Arguments {name}"
        }
        for name, param in sig.parameters.items()
    }
    tool_schema = {
        "name": fn.__name__,
        "description": (fn.__doc__ or '').strip().split('\n')[0],
        "parameters": {
            "type": "object",
            "properties": props,
            "required": [n for n, p in sig.parameters.items() if p.default is inspect._empty]
        }
    }
    return tool_schema

fns = [get_current_weather]
tool_schema = []
for tool_fn in fns:
    tool_schema.append(to_schema(tool_fn))

# tool_schema = to_schema(get_current_weather)
# print(type(tool_schema))
pprint(tool_schema)

[{'description': 'Getting the current weather for a given city.',
  'name': 'get_current_weather',
  'parameters': {'properties': {'city': {'description': 'Arguments city',
                                         'type': 'string'},
                                'unit': {'description': 'Arguments unit',
                                         'type': 'string'}},
                 'required': ['city'],
                 'type': 'object'}}]


In [7]:
# ---------------------------------------------------------
# Provide the tool schema to the model
# ---------------------------------------------------------
messages=[
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "system", "name": "tool_spec", "content": json.dumps(tool_schema)},
    {"role": "user", "content": "Is Boston warmer today or Seattle?"}
]

# MODEL = "gemma3:1b"
MODEL = "llama3.2:3b"
second_raw = client.chat.completions.create(
    model = MODEL,
    messages=messages,
    temperature=0.5
)
print(second_raw.choices[0].message.content)

TOOL_CALL:{"name": "get_current_weather", "args": {"city": "Boston", "unit": "F"}}; TOOL_CALL:{"name": "get_current_weather", "args": {"city": "Seattle", "unit": "F"}}


## LangChain for Tool Calling

In [8]:
# ---------------------------------------------------------
# Step 1: Define tools for LangChain
# ---------------------------------------------------------

from langchain.tools import tool


def get_current_weather(city: str, unit: str = "celsius") -> str:
    """Return the current weather as a human-readable sentence."""
    return f"It is 23 {unit[0].upper()} and sunny in {city}"

@tool
def get_weather(city: str) -> str:
    """Return the current weather as a human-readable sentence."""
    return get_current_weather

In [9]:
# ---------------------------------------------------------
# Step 2: Initialize the LangChain Agent
# ---------------------------------------------------------

from langchain_community.chat_models import ChatOllama
from langchain.agents import initialize_agent, Tool, AgentType

# MODEL = "gemma3:1b"
MODEL = "llama3.2:3b"
llm = ChatOllama(model=MODEL, temperature=0.1)
tools = [get_weather]

# agent = loop over llm
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent.invoke({"input": "Do I need an umbrella in a Boston today?"})

/tmp/ipykernel_36251/2756595791.py:10: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=MODEL, temperature=0.1)
/tmp/ipykernel_36251/2756595791.py:14: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`



> Entering new AgentExecutor chain...
Thought: To determine if it's raining, I should get the current weather for Boston.

Action: get_weather
Action Input: Boston

Observation: <function get_current_weather at 0x7aa591e0dee0>
Thought:Question: Do I need an umbrella in a Boston today?
Thought: The function returned is just a reference, so I should call it to get the actual weather.

Action: get_weather
Action Input: Boston

Observation: <function get_current_weather at 0x7aa591e0dee0>
Thought:Question: Do I need an umbrella in a Boston today?
Thought: The function returned is just a reference, so I should call it to get the actual weather.

Action: get_weather
Action Input: Boston

Observation: <function get_current_weather at 0x7aa591e0dee0>
Thought:Question: Do I need an umbrella in a Boston today?
Thought: To determine if it's raining, I should call the get_weather function with 'Boston' as input.

Action: get_weather
Action Input: Boston

Observation: <function get_current_weathe

{'input': 'Do I need an umbrella in a Boston today?',
 'output': 'Agent stopped due to iteration limit or time limit.'}

In [10]:
# ---------------------------------------------------------
# Step 1: Add a web search tool
# ---------------------------------------------------------

from ddgs import DDGS
from langchain.tools import tool

def search_web(query: str, max_results: int =10) -> str:
    """Return the top max_results web search results for query as a single formatted string. Uses
     DuckDuck's API"""
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=max_results):
            results.append(f"- {r['title']} - {r['href']}")
    return "\n".join(results)


@tool
def web_search(query: str) -> str:
    """Search the web for query and return concise results."""
    return search_web(query)

In [11]:
# ---------------------------------------------------------
# Step 2: Initialize the web-search agent
# ---------------------------------------------------------
from langchain.agents import initialize_agent, AgentType
from langchain.llms import OpenAI

MODEL = "llama3.2:3b"
llm = ChatOllama(model=MODEL, temperature=0.1)
tools = [web_search]
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [12]:
# ---------------------------------------------------------
# Step 3: Test your Ask-the-Web agent
# ---------------------------------------------------------
try:
    agent.invoke({"input": "Do I need an umbrella in a Boston today?"})
except Exception as e:
    print(e)



> Entering new AgentExecutor chain...


Impersonate 'chrome_129' does not exist, using 'random'


Thought: To determine if it's going to rain in Boston, I should check the current weather forecast.

Action: web_search
Action Input: "Boston weather today"
Observation: - Boston, MA Weather Forecast | AccuWeather - 
- Boston, MA Weather Forecast | AccuWeather - https://www.accuweather.com/en/us/boston/02108/weather-forecast/348735
- Boston, MA weather forecast – NBC10 Boston - https://www.nbcboston.com/weather/
- Boston, Massachusetts Weather - https://weather.com/weather/today/l/bf217d537cc1c8074ec195ce07fb74de3c1593caa6033b7c3be4645ccc5b01de
- 7-Day Forecast 42.36N 71.07W - National Weather ServiceBoston, MA - Local Weather Today, 10-Day Forecasts | US HarborsBoston Weather ForecastBoston, MA weather forecast – NBC10 BostonBoston, MA - Local Weather Today, 10-Day Forecasts | US HarborsBoston, MA Weather Forecast | AccuWeatherBoston, MA Weather Conditions | Weather Underground - https://forecast.weather.gov/zipcity.php?inputstring=Boston,MA
- Boston, MA - Local Weather Today, 10-Day 